# The Indian Itinerary Project — Exploratory Data Analysis

This notebook explores the final, cleaned, feature-engineered dataset produced by
`scripts/run_pipeline.py`. Run the pipeline first if `final_dataset/indian_itinerary.db`
doesn't exist yet:

```bash
python scripts/run_pipeline.py
```

**Contents**
1. Setup & data load
2. Data profiling recap
3. Missing values
4. Attraction type & significance distribution
5. Budget & fee analysis
6. Ratings & popularity
7. State-level comparison
8. Seasonality / climate analysis
9. Correlation analysis
10. Top destinations
11. Tag analysis


In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

DB_PATH = Path("../final_dataset/indian_itinerary.db")
conn = sqlite3.connect(DB_PATH)

attractions = pd.read_sql("SELECT * FROM vw_attractions_full", conn)
states = pd.read_sql("SELECT * FROM states", conn)
tags_map = pd.read_sql("SELECT * FROM vw_destination_tag_map", conn)
sim_ratings = pd.read_sql("SELECT * FROM vw_sim_destination_ratings", conn)

print(f"attractions: {attractions.shape}")
attractions.head()

## 2. Data profiling recap

Full profiling lives in `docs/DATASET_UNDERSTANDING.md`; this is a quick in-notebook recap.

In [ ]:
attractions.describe(include='all').T

## 3. Missing values

In [ ]:
missing = attractions.isna().mean().sort_values(ascending=False) * 100
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(8, max(4, len(missing) * 0.3)))
missing.plot(kind="barh", ax=ax, color="#c0392b")
ax.set_xlabel("% missing")
ax.set_title("Missing values by column")
plt.tight_layout()
plt.show()

## 4. Attraction type & significance distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
attractions["attraction_type"].value_counts().head(15).plot(kind="barh", ax=axes[0], color="#2980b9")
axes[0].set_title("Top 15 attraction types")
axes[0].invert_yaxis()

attractions["significance"].value_counts().head(15).plot(kind="barh", ax=axes[1], color="#27ae60")
axes[1].set_title("Top 15 significance categories")
axes[1].invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Budget & fee analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

attractions["budget_category"].value_counts().reindex(["Free", "Low", "Medium", "High", "Unknown"]).plot(
    kind="bar", ax=axes[0], color="#8e44ad"
)
axes[0].set_title("Attractions by budget category")
axes[0].set_ylabel("count")

sns.boxplot(data=attractions, x="popularity_tier", y="estimated_daily_cost_inr",
            order=["Niche", "Emerging", "Popular", "Iconic"], ax=axes[1])
axes[1].set_title("Estimated daily cost by popularity tier")
plt.tight_layout()
plt.show()

## 6. Ratings & popularity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

attractions["google_rating"].dropna().plot(kind="hist", bins=20, ax=axes[0], color="#f39c12")
axes[0].set_title("Distribution of Google ratings")
axes[0].set_xlabel("rating")

attractions["popularity_tier"].value_counts().reindex(["Niche", "Emerging", "Popular", "Iconic"]).plot(
    kind="bar", ax=axes[1], color="#16a085"
)
axes[1].set_title("Attractions by popularity tier")
plt.tight_layout()
plt.show()

## 7. State-level comparison

In [ ]:
state_summary = pd.read_sql("SELECT * FROM vw_state_summary ORDER BY attraction_count DESC", conn)

fig, ax = plt.subplots(figsize=(10, 8))
top15 = state_summary.head(15).sort_values("attraction_count")
ax.barh(top15["state_name"], top15["attraction_count"], color="#2c3e50")
ax.set_title("Top 15 states by number of attractions")
ax.set_xlabel("attractions")
plt.tight_layout()
plt.show()

state_summary.head(10)

## 8. Seasonality / climate analysis

In [ ]:
season_cols = ["summer_suitability", "winter_suitability", "monsoon_suitability"]
season_counts = attractions[season_cols].apply(lambda s: s.value_counts()).fillna(0)

season_counts.T.plot(kind="bar", stacked=True, figsize=(10, 6),
                      color=["#27ae60", "#f39c12", "#c0392b"])
plt.title("Seasonal suitability of destination cities")
plt.ylabel("count of attractions")
plt.legend(title="Suitability")
plt.tight_layout()
plt.show()

## 9. Correlation analysis

In [ ]:
numeric_cols = [
    "google_rating", "entrance_fee_inr", "visit_duration_hrs",
    "adventure_score", "photography_score", "culture_score", "history_score",
    "nature_score", "pilgrimage_score", "accessibility_score", "estimated_daily_cost_inr",
]
corr = attractions[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation matrix — numeric & engineered features")
plt.tight_layout()
plt.show()

## 10. Top destinations

In [ ]:
top_by_reviews = attractions.sort_values("google_review_count_lakhs", ascending=False).head(15)
top_by_reviews[["attraction_name", "city_name", "state_name", "google_rating", "google_review_count_lakhs"]]

In [ ]:
# Synthetic domain: top rated sim_destinations (min 3 reviews) — see docs/DATASET_UNDERSTANDING.md
sim_ratings[sim_ratings["review_count"] >= 3].sort_values("avg_review_rating", ascending=False).head(15)

## 11. Tag analysis

In [ ]:
tag_counts = tags_map["tag_name"].value_counts()

fig, ax = plt.subplots(figsize=(10, 8))
tag_counts.sort_values().plot(kind="barh", ax=ax, color="#34495e")
ax.set_title("Attraction counts by tag")
plt.tight_layout()
plt.show()

## Summary of findings

- Attraction density is heavily concentrated in a handful of states
  (Maharashtra, Uttar Pradesh, West Bengal, Delhi, Karnataka lead the count).
- Entrance fees skew low: most attractions fall into the Free/Low budget
  categories, consistent with India's public-heritage-site pricing.
- Climate-suitability coverage is limited to the ~64/213 cities with a
  matched temperature record — a clear enrichment target in `docs/ROADMAP.md`.
- The two data domains (real attractions vs. synthetic user-engagement data)
  behave very differently, which is expected and documented — they should
  not be compared as if measuring the same phenomenon.

See `docs/DATASET_UNDERSTANDING.md` and `docs/CLEANING_LOG.md` for full
methodology and every cleaning decision made upstream of this notebook.

In [ ]:
conn.close()